In [ ]:
from time import sleep
from pynq.overlays.base import BaseOverlay
from pynq import GPIO
import pynq
import subprocess as sp
import netifaces as ni
import socket
import struct
import numpy as np
from doa_mts import *
import scipy.io
import os
o1 = doaMtsOverlay('doa_mts.bit')

In [ ]:
# --- Cell 1: TCP(JSON) + UDP streaming bridge for your existing o1 device ---
import asyncio, json, struct, socket, os, time, contextlib
from dataclasses import dataclass, asdict

# Optional stubs to avoid NameErrors if you use ps_btn/oled
class _StubBtn:
    def read(self): return 0
class _StubOled:
    def write(self, s): 
        try: print("[OLED]", s)
        except: pass

try:
    ps_btn
except NameError:
    ps_btn = _StubBtn()
try:
    oled
except NameError:
    oled = _StubOled()

@dataclass
class BridgeConfig:
    dest_ip: str = "127.0.0.1"   # learned from TCP peer
    dest_port: int = 5005        # MATLAB udpport LocalPort
    send_interval_s: float = 0.0 # 0 = as fast as get_custom_data returns
    # (If you prefer rate-based, set to e.g. 0.01 for ~100 Hz outer loop)

class TxRxBridge:
    """
    - TCP server (JSON per line) for commands
    - Learns MATLAB IP from TCP
    - Keeps listening for JSON commands while streaming
    - UDP TX loop packs o1.get_custom_data() as int16 and sends continuously when o1.dataStream==1
    - Runs until stop() is called
    """
    def __init__(self, o1, oled=oled):
        self.o1 = o1
        self.cfg = BridgeConfig()
        self._tcp_server = None
        self._udp_sock = None
        self._udp_target = None # (ip, port)
        self._sender_task = None
        self._run_event = asyncio.Event()
        self._client_task = None
        self._client_addr = None
        self._stop_flag = False
        self.oled = oled

        # ensure required o1 fields exist
        if not hasattr(self.o1, "dataStream"):
            self.o1.dataStream = 0
        if not hasattr(self.o1, "data_size"):
            self.o1.data_size = 1024

    # ---------- Public controls ----------
    async def start(self, host="0.0.0.0", port=9000):
        # TCP server with address reuse
        if self._tcp_server:
            return self._tcp_server
        self._tcp_server = await asyncio.start_server(
            self._handle_client, host, port,
            reuse_address=True, reuse_port=False
        )
        addrs = ", ".join(str(s.getsockname()) for s in self._tcp_server.sockets)
        print(f"[TCP] listening on {addrs}")
        # Start UDP sender task (idle until dataStream=1)
        if not self._sender_task or self._sender_task.done():
            self._sender_task = asyncio.create_task(self._udp_sender_loop())
        return self._tcp_server

    async def stop(self):
        self._stop_flag = True
        # stop UDP task
        if self._sender_task:
            self._sender_task.cancel()
            with contextlib.suppress(asyncio.CancelledError):
                await self._sender_task
            self._sender_task = None
        # close UDP socket
        if self._udp_sock:
            try: self._udp_sock.close()
            except: pass
            self._udp_sock = None
            self._udp_target = None
        # stop TCP
        if self._tcp_server:
            self._tcp_server.close()
            await self._tcp_server.wait_closed()
            self._tcp_server = None
            print("[TCP] server stopped")

    # ---------- TCP handling ----------
    async def _handle_client(self, reader: asyncio.StreamReader, writer: asyncio.StreamWriter):
        peer = writer.get_extra_info("peername")  # (ip, port)
        client_ip = peer[0] if peer else "127.0.0.1"
        self._client_addr = peer

        # Learn MATLAB IP
        self.cfg.dest_ip = client_ip
        self._udp_target = (self.cfg.dest_ip, self.cfg.dest_port)
        self._ensure_udp_socket()

        await self._send_json(writer, {
            "ok": True,
            "msg": f"connected from {peer}",
            "udp_dest": {"ip": self.cfg.dest_ip, "port": self.cfg.dest_port},
            "cfg": asdict(self.cfg),
        })
        self.oled.write(f"TCP client {client_ip} connected")

        try:
            while not self._stop_flag:
                # graceful shutdown via physical button (optional)
                if hasattr(ps_btn, "read") and ps_btn.read() == 1:
                    self.oled.write("PS BTN reset\nServer Shutdown")
                    await self._send_json(writer, {"ok": True, "msg": "shutdown"})
                    await self.stop()
                    return

                # read one JSON line
                line = await reader.readline()
                if not line:
                    break  # client closed
                line = line.decode().strip()
                if not line:
                    continue

                # parse JSON (or ignore)
                try:
                    cmd = json.loads(line)
                except json.JSONDecodeError:
                    await self._send_json(writer, {"ok": False, "error": "invalid_json"})
                    continue

                # Apply the command set
                resp = await self._dispatch_json(cmd)
                await self._send_json(writer, resp)

                # allow 'bye' to close just the session (server stays alive)
                if (cmd.get("cmd") or "").lower() == "bye":
                    break
        finally:
            writer.close()
            with contextlib.suppress(Exception):
                await writer.wait_closed()
            self.oled.write("TCP client disconnected")
            self._client_addr = None

    async def _send_json(self, writer, obj):
        writer.write((json.dumps(obj) + "\n").encode())
        await writer.drain()

    # ---------- Command mapping (JSON -> your legacy actions) ----------
    async def _dispatch_json(self, cmd: dict):
        """
        Expected shapes (JSON per line):
        {"cmd":"config", "dest_port":5005, "pkt_size":1024, "send_interval_ms":0, "payload":"counter"}
        {"cmd":"dataStream", "value":1}
        {"cmd":"fc", "vals":[fc, nz, d_fc0, d_nz0, d_fc1, d_nz1]}      # like your 'fc' branch
        {"cmd":"cal", "channels":N}
        {"cmd":"phase", "vals":[...]}
        {"cmd":"dphase", "vals":[...]}
        {"cmd":"da", "value":0|1|2}
        {"cmd":"dac0", "vals":[...]}
        {"cmd":"dac1", "vals":[...]}
        {"cmd":"dacPow", "vals":[...]}
        {"cmd":"status"}
        {"cmd":"bye"}
        """
        c = (cmd.get("cmd") or "").lower()

        try:
            if c == "config":
                # enforce learned IP; accept dest_port, pkt_size, send_interval
                if "dest_port" in cmd:
                    self.cfg.dest_port = int(cmd["dest_port"])
                if "pkt_size" in cmd:
                    self.o1.data_size = int(cmd["pkt_size"])
                if "send_interval_ms" in cmd:
                    ms = float(cmd["send_interval_ms"])
                    self.cfg.send_interval_s = max(0.0, ms/1000.0)
                if "payload" in cmd:
                    # optional: allow "zeros"|"random"|"counter" as a software payload,
                    # but your real data comes from o1; we keep for completeness
                    pass
                # re-target UDP if port changed
                self._udp_target = (self.cfg.dest_ip, self.cfg.dest_port)
                self._ensure_udp_socket()
                # Auto-start sender loop (it’s always running; data gated by dataStream)
                return {"ok": True, "cfg": asdict(self.cfg), "o1": {"data_size": self.o1.data_size, "dataStream": self.o1.dataStream}}

            if c == "datastream":
                val = int(cmd.get("value", 0))
                self.o1.dataStream = 1 if val != 0 else 0
                self.oled.write(f"DataStream: {self.o1.dataStream}")
                return {"ok": True, "dataStream": self.o1.dataStream}

            if c == "fc":
                vals = list(map(int, cmd.get("vals", [])))
                if len(vals) >= 6:
                    self.o1.centre_freq, self.o1.nyquist_zone = vals[0], vals[1]
                    self.o1.d_centre_freq = [vals[2], vals[2], vals[4], vals[4]]
                    self.o1.d_nyquist_zone = [vals[3], vals[3], vals[5], vals[5]]
                    self.o1.configure_dacs(); self.o1.configure_adcs(); self.o1.sync_tiles()
                return {"ok": True}

            if c == "cal":
                ch = int(cmd.get("channels", 0))
                self.o1.channels = ch
                self.o1.calibrate()
                return {"ok": True}

            if c == "phase":
                vals = list(map(int, cmd.get("vals", [])))
                self.o1.phases = vals
                self.o1.configure_adcs(); self.o1.sync_tiles()
                return {"ok": True}

            if c == "dphase":
                vals = list(map(int, cmd.get("vals", [])))
                self.o1.d_phases = vals
                self.o1.configure_dacs(); self.o1.sync_tiles()
                return {"ok": True}

            if c == "datachan":
                n = int(cmd.get("value", self.o1.data_size))
                self.o1.data_size = n
                self.oled.write(f"Data Size:\n{self.o1.data_size}")
                return {"ok": True, "data_size": self.o1.data_size}

            if c == "da":
                self.o1.da = int(cmd.get("value", 0))
                if self.o1.da == 2:
                    self.o1.dacs_off(); self.o1.trig_cap.off(); self.o1.trig_cap.on(); self.o1.trig_cap.off()
                elif self.o1.da == 1:
                    self.o1.dacs_off(); self.o1.trig_cap.off(); self.o1.trig_cap.on(); self.o1.dacs_control(); self.o1.trig_cap.off()
                else:
                    self.o1.dacs_off(); self.o1.dacs_control()
                return {"ok": True}

            if c == "dac0":
                vals = list(map(int, cmd.get("vals", [])))
                self.o1.dac_signal[0] = vals
                self.oled.write(f"Dac0 loading:\n{len(vals)}")
                self.o1.dac_data_mem_write(self.o1.dac_signal[0], self.o1.dac0_player)
                self.oled.write("Dac0 loaded")
                return {"ok": True, "count": len(vals)}

            if c == "dac1":
                vals = list(map(int, cmd.get("vals", [])))
                self.o1.dac_signal[2] = vals
                self.oled.write(f"Dac1 loading:\n{len(vals)}")
                self.o1.dac_data_mem_write(self.o1.dac_signal[2], self.o1.dac1_player)
                self.oled.write("Dac1 loaded")
                return {"ok": True, "count": len(vals)}

            if c == "dacpow":
                vals = list(map(int, cmd.get("vals", [])))
                self.o1.dacPower = vals
                return {"ok": True}

            if c == "status":
                return {"ok": True, "cfg": asdict(self.cfg),
                        "udp_target": self._udp_target,
                        "dataStream": getattr(self.o1, "dataStream", 0),
                        "data_size": getattr(self.o1, "data_size", None)}

            if c == "bye":
                # keep server alive; client can reconnect
                return {"ok": True, "status": "closing"}

            return {"ok": False, "error": "unknown_cmd", "cmd": c}
        except Exception as e:
            # If your device methods raise, report back
            return {"ok": False, "error": f"{type(e).__name__}: {e}"}

    # ---------- UDP sender (continuous) ----------
    def _ensure_udp_socket(self):
        target = (self.cfg.dest_ip, self.cfg.dest_port)
        if self._udp_sock is None:
            self._udp_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self._udp_target = target

    async def _udp_sender_loop(self):
        """
        Runs until cancelled. If o1.dataStream == 1, continuously:
          - fetch raw data (int16) from o1.get_custom_data(data_size)
          - pack to binary and send via UDP to learned MATLAB IP:dest_port
        """
        print("[UDP] sender task running")
        try:
            while True:
                if getattr(self.o1, "dataStream", 0) != 1:
                    await asyncio.sleep(0.01)
                    continue

                # Make sure UDP target is ready
                if not self._udp_sock or not self._udp_target:
                    await asyncio.sleep(0.01)
                    continue

                n = int(getattr(self.o1, "data_size", 0) or 0)
                if n <= 0:
                    await asyncio.sleep(0.005)
                    continue

                # --- Acquire & pack (your original logic preserved) ---
                data = self.o1.get_custom_data(n)  # expects np.int16 array/list length n
                # defensive checks
                if hasattr(data, "__len__") and len(data) >= n:
                    # pack int16
                    try:
                        payload = struct.pack('h' * n, *data[:n])
                    except Exception as e:
                        # Fallback: convert to int first
                        arr = [int(x) for x in data[:n]]
                        payload = struct.pack('h' * n, *arr)
                    # send
                    try:
                        self._udp_sock.sendto(payload, self._udp_target)
                    except OSError as e:
                        # e.g., target unreachable; wait a bit
                        await asyncio.sleep(0.02)
                else:
                    await asyncio.sleep(0.001)

                # pacing (optional)
                if self.cfg.send_interval_s > 0.0:
                    await asyncio.sleep(self.cfg.send_interval_s)
        finally:
            print("[UDP] sender task stopped")
            if self._udp_sock:
                try: self._udp_sock.close()
                except: pass
                self._udp_sock = None
                self._udp_target = None

# Expose a singleton “bridge” for convenience
# IMPORTANT: define/create your device 'o1' BEFORE creating the bridge.
try:
    BRIDGE  # do we already have one?
except NameError:
    # You MUST have created o1 elsewhere. If not, raise a helpful error.
    try:
        o1  # noqa
    except NameError as _e:
        raise RuntimeError("Please create your acquisition object 'o1' before running this cell.") from _e
    BRIDGE = TxRxBridge(o1=o1, oled=oled)


In [ ]:
# --- Cell 2: Start the TCP server. It will keep running until you stop it. ---
await BRIDGE.start(host=os.environ["RFSOC_BIND_IP"], port=9000)

# When you want to stop everything:
# await BRIDGE.stop()
